# KXHIGHLAX Single-Market Mispricing Episodes

For every locally available day, finds the moments where ONE specific listed market's own quoted `[bid, ask]` diverged furthest from the fitted curve's own price for it (`black_scholes.smile_bucket_prob` off that snapshot's already-fitted `implied_spot`/`implied_sigma`/wing pillars -- the same curve `analysis.ipynb`'s snapshot diagnostics draw), and how long it took to converge back.

**Data source: Kalshi's own live L2 websocket order book** (`plotting.load_kalshi_ws_long_df`), not Predexon. Two real problems with the Predexon-sourced feed were confirmed directly: (1) genuine crossed-book data (bid > ask -- e.g. `KXHIGHLAX-26FEB22` had three markets simultaneously crossed, one by 13 cents) that fed spurious "mispricing episodes", and (2) far slower convergence (30min-7h) than economically sensible for a real cross-market mispricing. Kalshi's own exchange-side book is the more authoritative source -- at the cost of only covering however far back the WS collector has been running (`plotting.list_available_kalshi_ws_days`), far short of Predexon's multi-month archive.

**Zero live network calls.** `load_kalshi_ws_long_df` is a plain local parquet read (the WS collector's own output, backfilled from S3 for any day already closed and finalized there); `compute_implied_spot_timeseries_cached` is called with an explicit EMPTY `kalshi_backup` so its own internal Kalshi-1-min-candle fallback -- normally a live API call -- never fires (long_df is already Kalshi's own data, so there's nothing to fall back to). `staleness_threshold` is widened to 10 minutes (vs. the 90s default tuned for Predexon's much denser tick stream): the WS collector only logs a new quote when the top of book actually changes, so a quiet book can legitimately go several minutes with no new tick -- that's not staleness, just an unchanged price.

**Only live order-book data counts.** A market's own quote only contributes here while it's *live* -- i.e. no quote older than `staleness_threshold` and not a crossed (bid > ask) reading (`black_scholes.market_live_matrix`). Both a mispricing episode's onset (deviation first crosses `onset_threshold`) and its fix (deviation first returns under `fixed_threshold`) must fall within the *same* unbroken live run for that market -- an episode still open when the live run breaks is dropped rather than reported with an inferred fix time.

For each qualifying episode (ranked by peak |deviation| reached, up to `top_n`), plots a before/after pair: every listed market's own bid/ask alongside the fitted curve at the mispricing's onset, and again at the moment it was fixed -- see `plotting.plot_mispricing_episode` / `plotting._render_snapshot_panel`. With only 16 days of kalshi_ws history available, expect FEWER than `top_n` genuine episodes to qualify (confirmed: 3, all correcting within a single 1-minute sample) -- that's the real, clean answer for this window, not a bug.

In [ ]:
import sys
import os
import importlib
_NB_DIR = os.path.dirname(os.path.abspath('__file__'))
sys.path.insert(0, _NB_DIR)
sys.path.insert(0, os.path.join(_NB_DIR, "..", "kalshi_infra"))

import datetime
from datetime import timezone, timedelta

import api_functions
import black_scholes
import plotting
importlib.reload(api_functions)
importlib.reload(black_scholes)
importlib.reload(plotting)

from api_functions import event_ticker_from_date
from plotting import LA_TZ

In [ ]:
def day_window_ts(date):
    """
    (start_ts, end_ts) covering the full market lifetime in PT -- previous PT
    midnight -> next PT midnight (48h), same convention analysis.ipynb uses.
    """
    event_start_pt = LA_TZ.localize(datetime.datetime(date.year, date.month, date.day))
    start = event_start_pt - timedelta(days=1)
    end = event_start_pt + timedelta(days=1)
    now = int(datetime.datetime.now(timezone.utc).timestamp())
    return int(start.timestamp()), min(int(end.timestamp()), now)


# Widened from compute_implied_spot_timeseries's own 90s default (tuned for
# Predexon's much denser tick stream) -- the WS collector only logs a new
# quote when the top of book actually changes, so a quiet book can
# legitimately go several minutes with no new tick. Used both when fitting
# each day below (against an EMPTY kalshi_backup, so it's the only liveness
# check that applies -- see the intro markdown) and later when finding
# mispricing episodes, so both stages agree on what "live" means.
STALENESS_THRESHOLD = '10min'
KALSHI_WS_CACHE_DIR = os.path.join(_NB_DIR, 'data', 'implied_ts_kalshi_ws')


def fetch_day_timeseries(date, freq='1min'):
    """
    Implied spot/sigma for one day at `freq`, fit ENTIRELY off Kalshi's own
    live L2 order book (plotting.load_kalshi_ws_long_df) -- cached, so a
    zero-network local parquet read on a cache miss and a zero-network
    parquet read on a cache hit either way. kalshi_backup is passed as an
    explicit EMPTY DataFrame so compute_implied_spot_timeseries's own
    internal staleness/mismatch fallback -- normally a live Kalshi 1-min-
    candle API call -- never fires; long_df is already Kalshi's own data, so
    there's nothing to fall back to.
    """
    event_ticker = event_ticker_from_date(date)
    start_ts, end_ts = day_window_ts(date)
    ts_df = black_scholes.compute_implied_spot_timeseries_cached(
        lambda: plotting.load_kalshi_ws_long_df(event_ticker),
        start_ts, end_ts, event_ticker, freq=freq,
        cache_dir=KALSHI_WS_CACHE_DIR, fit_smile=True,
        staleness_threshold=STALENESS_THRESHOLD, kalshi_backup=plotting._EMPTY_KALSHI_BACKUP,
    )
    print(f'{event_ticker} ({date}): {len(ts_df)} time points ({freq} grid)')
    return event_ticker, ts_df, date

## Day selection

Every locally available kalshi_ws day (`plotting.list_available_kalshi_ws_days` -- a plain local directory scan, no network call at all), fetched at a 1-minute grid. Far fewer days than the Predexon archive (see intro), but the expensive per-market gather step below is cached to parquet (`plotting._gather_market_deviation_table`), so reruns are instant either way.

In [ ]:
FREQ = '1min'

AVAILABLE_DAYS = plotting.list_available_kalshi_ws_days('KXHIGHLAX')
print(f'{len(AVAILABLE_DAYS)} kalshi_ws days locally, {AVAILABLE_DAYS[0]} .. {AVAILABLE_DAYS[-1]}')

day_results = [fetch_day_timeseries(d, freq=FREQ) for d in AVAILABLE_DAYS]

## Top mispricing episodes

`onset_threshold`: how far outside its own quoted band the fitted curve's price has to sit before a stretch counts as "mispriced" (in probability/price units, e.g. 0.02 = the model prices this market 2 cents richer/cheaper than its own best bid/ask). `fixed_threshold`: how close back to the band counts as "fixed" (tighter than onset, so a episode doesn't flicker closed the instant it barely re-enters the band). `warmup` skips each event's own thin, staggered-opening minutes. `staleness_threshold` must match the one used to fit `day_results` above (see `STALENESS_THRESHOLD`) -- this is a SEPARATE liveness check (over the raw kalshi_ws bid/ask, via `black_scholes.market_live_matrix`) from the one the fit itself used, and the two would silently disagree otherwise. `data_source='kalshi_ws'` selects the same zero-network loader used above for rendering each episode's onset/fixed snapshot. Capped to at most ONE episode per (event, market) -- its own best -- so a single market that swings in and out of mispricing repeatedly in one day can't fill most of `top_n` by itself; see `plotting.find_market_mispricing_episodes`'s docstring for the exact selection logic.

With only 16 days of kalshi_ws history, expect FEWER than `top_n` genuine episodes -- confirmed: 3, all correcting within a single 1-minute sample.

In [ ]:
mispricings = plotting.plot_market_mispricings(
    day_results,
    top_n=10,
    onset_threshold=0.02,
    fixed_threshold=0.002,
    min_duration='1min',
    warmup='15min',
    staleness_threshold=STALENESS_THRESHOLD,
    data_source='kalshi_ws',
)